# Open-model labeler (Google Colab)
Labels news and Reddit items with a **free open-weights model** using exactly the same prompt, fields and allowed values as the main notebook (Part 4). Nothing here is paid, and the model name and weights are fixed, so anyone can rerun it.

**Before you start (once)**
1. In the main notebook, run Part 4 up to and including section 4.8. It writes four files into `data/processed/`:
   `day4_ai_prompt.txt`, `colab_input_sample300.csv`, `colab_input_swap.csv`, `colab_input_corpus.csv`.
2. In Google Drive, create a folder called `fre7871_labels` and upload those four files into it.
3. In Colab: **Runtime > Change runtime type > GPU** (A100 if you have it, otherwise L4), then **Run all**. The first run of the GPU step downloads about 15 GB of model weights.

**Order of work** (each step is cached on Drive, so a disconnect only costs the unfinished items; just run again):
trial of 20 items -> the 300-item sample (you check agreement with your human labels in the main notebook, section 4.9) -> the swap-test items -> the full corpus (about 20,000 items).

**Outputs** (written to the same Drive folder, copy them into `data/processed/`):
`day4_ai_labels_OPEN_sample.csv`, `day4_ai_labels_OPEN_swap.csv`, `day4_ai_labels_OPEN_corpus.csv`.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [2]:
# Install vLLM (a fast inference server). This takes a few minutes. If Colab asks you to restart the runtime, do it, then run from the top again.
!pip -q install vllm
import vllm
print("vLLM version:", vllm.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72

In [3]:
import os, re, json, time, hashlib, subprocess, threading, datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import requests
import pandas as pd

# ---------- Settings (fixed before any result is seen) ----------
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct-AWQ"      # open weights, Apache 2.0. A larger alternative is "Qwen/Qwen2.5-14B-Instruct" (needs an A100).
                                              # 14B on an L4 (24 GB): MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct-AWQ" and QUANT = "awq". 14B full precision needs an A100.
QUANT = "awq"                                  # set to "awq" ONLY together with an "-AWQ" model name (4-bit; fits smaller GPUs)
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1/chat/completions"
WORKERS = 32                                  # parallel requests; vLLM batches them on the GPU
MAX_TOKENS = 120
SEED = 7

try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/fre7871_labels")
except ImportError:                           # running outside Colab
    WORK = Path("fre7871_labels")
WORK.mkdir(exist_ok=True); (WORK / "cache").mkdir(exist_ok=True)
def check_drive():
    """Drive can disconnect after a runtime restart. Stop with a clear message instead of silently writing results to a temporary folder."""
    if not (WORK / "day4_ai_prompt.txt").exists():
        raise RuntimeError(f"Cannot see {WORK}/day4_ai_prompt.txt. Google Drive is probably disconnected: re-run THIS cell (it re-mounts Drive), "
                           "and check that the folder is named fre7871_labels and holds the four input files.")
check_drive()
print("Working folder:", WORK, "| files there:", sorted(p.name for p in WORK.iterdir() if p.is_file()))

Mounted at /content/drive
Working folder: /content/drive/MyDrive/fre7871_labels | files there: ['colab_input_corpus.csv', 'colab_input_sample300.csv', 'colab_input_swap.csv', 'day4_ai_labels_OPEN_sample.csv', 'day4_ai_labels_OPEN_swap.csv', 'day4_ai_prompt.txt']


In [4]:
# ---------- Prompt and schema: read from the main notebook's file so they cannot drift apart ----------
raw = (WORK / "day4_ai_prompt.txt").read_text(encoding="utf-8")
header, SYSTEM_PROMPT = raw.split("\n\n", 1)
SYSTEM_PROMPT = SYSTEM_PROMPT.strip()
PROMPT_HASH = hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:8]
claimed = re.search(r"PROMPT_HASH:\s*(\w+)", header).group(1)
assert PROMPT_HASH == claimed, f"prompt file is damaged or edited: hash {PROMPT_HASH} != {claimed}"
print("Prompt hash", PROMPT_HASH, "(matches the main notebook)")

ALLOWED = {"topic": ["prices_inflation", "jobs_wages", "tariffs_trade", "rates_housing", "growth_recession", "other_econ"],
           "frame": ["decline_crisis", "cost_burden", "resilience_recovery", "risk_uncertainty", "political_blame", "neutral_report"],
           "blame": ["none", "trump_gop", "biden_dems", "fed", "business", "foreign", "other"]}
def _enum(vals): return {"type": ["string", "null"], "enum": vals + [None]}
SCHEMA = {"type": "object", "additionalProperties": False,
          "required": ["us_econ", "topic", "tone", "frame", "blame", "confidence"],
          "properties": {"us_econ": {"type": "integer", "enum": [0, 1]},
                         "topic": _enum(ALLOWED["topic"]), "frame": _enum(ALLOWED["frame"]), "blame": _enum(ALLOWED["blame"]),
                         "tone": {"type": ["integer", "null"], "enum": [-2, -1, 0, 1, 2, None]},
                         "confidence": {"type": "integer", "enum": [1, 2, 3]}}}

def validate(lab):
    """True only if the label is complete and uses allowed values. us_econ = 0 must leave the four content fields empty; us_econ = 1 must fill them."""
    try:
        if set(lab) != set(SCHEMA["required"]): return False
        if lab["us_econ"] not in (0, 1) or lab["confidence"] not in (1, 2, 3): return False
        fields = (lab["topic"], lab["tone"], lab["frame"], lab["blame"])
        if lab["us_econ"] == 0: return all(v is None for v in fields)
        return (lab["topic"] in ALLOWED["topic"] and lab["frame"] in ALLOWED["frame"] and lab["blame"] in ALLOWED["blame"]
                and lab["tone"] in (-2, -1, 0, 1, 2) and not isinstance(lab["tone"], bool))
    except Exception:
        return False

CONTENT = ("topic", "tone", "frame", "blame")
def normalize(lab):
    """AMENDMENT 2 (format only, fixed before the next run): if the model says us_econ = 0 but still fills topic/tone/frame/blame,
    those four fields are set to null, because the analysis never uses them for items that are not about the US economy.
    An answer with us_econ = 1 and missing fields is NOT repaired and stays invalid. Returns (label, was_repaired)."""
    if isinstance(lab, dict) and lab.get("us_econ") == 0 and any(lab.get(c) is not None for c in CONTENT):
        return {**lab, **{c: None for c in CONTENT}}, True
    return lab, False

def item_text(r):
    f = lambda v: "-" if (v is None or (isinstance(v, float) and v != v) or str(v).strip() == "") else str(v)
    return f"HEADLINE: {f(r['title'])}\nSTANDFIRST: {f(r['standfirst'])}\nEXCERPT: {f(r['text_excerpt'])}"

Prompt hash 7bd722c0 (matches the main notebook)


In [5]:
# ---------- Start the model server (downloads the weights the first time; allow 5 to 15 minutes) ----------
LOG = Path("/tmp/vllm_server.log")                  # local disk on purpose: it must not depend on Drive being connected
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_NAME, "--port", str(PORT), "--dtype", "auto",
     "--max-model-len", "4096", "--gpu-memory-utilization", "0.90", "--enable-prefix-caching", "--seed", str(SEED)] + (["--quantization", QUANT] if QUANT else []),
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM exited. Last log lines:\n" + "\n".join(LOG.read_text().splitlines()[-25:]))
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=3).status_code == 200: break
    except Exception:
        pass
    if time.time() - t0 > 1500: raise TimeoutError("Server not ready after 25 minutes; see " + str(LOG))
    time.sleep(10)
print(f"Server ready after {time.time() - t0:.0f} s. Model: {MODEL_NAME}")

Server ready after 290 s. Model: Qwen/Qwen2.5-14B-Instruct-AWQ


In [6]:
# ---------- Labeler: one call per item, temperature 0, JSON restricted to the schema, every answer cached ----------
_lock = threading.Lock()
_mode = {"response_format": True}                       # falls back to the older `guided_json` option if the server rejects response_format

def _call(text):
    for attempt in range(5):
        body = {"model": MODEL_NAME, "temperature": 0, "seed": SEED, "max_tokens": MAX_TOKENS,
                "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]}
        if _mode["response_format"]:
            body["response_format"] = {"type": "json_schema", "json_schema": {"name": "label", "strict": True, "schema": SCHEMA}}
        else:
            body["guided_json"] = SCHEMA
        try:
            r = requests.post(BASE_URL, json=body, timeout=120)
        except Exception as e:
            last = str(e)[:150]; time.sleep(2 ** attempt); continue
        if r.status_code == 200:
            return r.json()["choices"][0]["message"]["content"]
        if r.status_code == 400 and _mode["response_format"] and "response_format" in r.text:
            _mode["response_format"] = False; continue
        last = f"HTTP {r.status_code}: {r.text[:150]}"; time.sleep(2 ** attempt)
    raise RuntimeError("request failed: " + last)

def label_items(items, tag, limit=None):
    """items: DataFrame with key, title, standfirst, text_excerpt. Returns a DataFrame of labels. Resumable: finished items are read from the cache."""
    check_drive()
    path = WORK / "cache" / f"ai_{MODEL_NAME.replace('/', '_')}_{PROMPT_HASH}_{tag}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip(): d = json.loads(line); done[d["key"]] = d
    for d in done.values():                              # re-check earlier invalid answers under the amendment (the raw text is stored)
        if not d["ok"] and d.get("raw"):
            try: lab2, rep2 = normalize(json.loads(d["raw"]))
            except Exception: continue
            if validate(lab2): d.update(ok=True, label=lab2, repaired=True, raw=None)
    todo = [r for _, r in items.iterrows() if r["key"] not in done]
    if limit: todo = todo[:limit]
    print(f"{tag}: {len(done)} cached, {len(todo)} to label")
    t0 = time.time(); count = [0]
    def work(r):
        txt = item_text(r)
        raw = _call(txt); lab, ok, rep = None, False, False
        for _ in range(2):                              # if the output is not valid, ask once more (same settings; the server may differ slightly under load)
            try: lab, rep = normalize(json.loads(raw)); ok = validate(lab)
            except Exception: lab, ok, rep = None, False, False
            if ok: break
            raw = _call(txt)
        rec = {"key": r["key"], "ok": bool(ok), "repaired": bool(ok and rep), "label": lab if ok else None, "raw": None if ok else str(raw)[:300],
               "utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
        with _lock:
            with path.open("a", encoding="utf-8") as fh: fh.write(json.dumps(rec) + "\n")
            done[r["key"]] = rec; count[0] += 1
            if count[0] % 500 == 0: print(f"  {count[0]}/{len(todo)} done, {count[0] / (time.time() - t0):.1f} items/s", flush=True)
    if todo:
        with ThreadPoolExecutor(WORKERS) as ex: list(ex.map(work, todo))
        print(f"  finished {len(todo)} in {time.time() - t0:.0f} s")
    rows = []
    for k, d in done.items():
        lab = d["label"] or {}
        rows.append({"key": k, "valid": int(d["ok"]), "repaired": int(d.get("repaired", False)), **{f"ai_{c}": ("" if lab.get(c) is None else lab[c])
                     for c in ("us_econ", "topic", "tone", "frame", "blame", "confidence")}})
    out = pd.DataFrame(rows)
    out["ai_model"] = MODEL_NAME; out["ai_prompt_hash"] = PROMPT_HASH
    return out

def load_items(name):
    df = pd.read_csv(WORK / name, dtype=str).fillna("")
    assert {"key", "title", "standfirst", "text_excerpt"} <= set(df.columns) and df.key.is_unique, f"{name}: unexpected format"
    return df

## Trial: 20 items
Read these by eye. Do the labels look sensible? Is every item `valid = 1`? If most are invalid or all identical, stop and tell me before running the rest.

In [7]:
s300 = load_items("colab_input_sample300.csv")
trial = label_items(s300.head(20), "trial20")
show = trial.merge(s300[["key", "title"]], on="key")
pd.set_option("display.width", 250, "display.max_colwidth", 70)
print("valid:", int(trial.valid.sum()), "of", len(trial))
show[["key", "title", "ai_us_econ", "ai_topic", "ai_tone", "ai_frame", "ai_blame", "ai_confidence"]]

trial20: 0 cached, 20 to label
  finished 20 in 4 s
valid: 20 of 20


,key,title,ai_us_econ,ai_topic,ai_tone,ai_frame,ai_blame,ai_confidence
0,L001,Takeaways: Supreme Court stands up to Donald Trump on emergency ta...,0,,,,,3
1,L003,China population: ‘extremely low’ birth rate in Wuhu highlights na...,0,,,,,3
2,L006,"""The first lesson of economics is scarcity: There is never enough ...",0,,,,,3
3,L017,Education Sector In Telangana | Part-3 | Economy | Sujath | Tone A...,0,,,,,3
4,L018,Trump news at a glance: six in 10 Americans say president is doing...,0,,,,,3
5,L012,"How Democrats Built China’s Economy, From Clinton To Biden",0,,,,,3
6,L002,US pauses sanctions on Iranian oil ‘stranded at sea’ to help curb ...,1,other_econ,0,neutral_report,none,3
7,L004,IMF says fragmentation could cost global economy up to 7% of GDP,0,,,,,3
8,L015,Mortgage Help for first time buyer,1,rates_housing,0,neutral_report,none,3
9,L007,Fed could cut interest rates again as US inflation barely moves in...,1,rates_housing,-1,risk_uncertainty,none,3


## Step 1: the 300-item sample
After this finishes, copy `day4_ai_labels_OPEN_sample.csv` into `data/processed/` and run section 4.9 of the main notebook. **Only continue to the full corpus if that section says the labeler passes.**

In [8]:
def save(df, name, idcol):
    out = df.rename(columns={"key": idcol})
    out.to_csv(WORK / name, index=False, encoding="utf-8")
    bad = int((out.valid == 0).sum())
    print(f"Saved {name}: {len(out)} rows, {bad} invalid ({bad / max(len(out), 1):.1%}), {int(out.repaired.sum())} repaired (us_econ=0 with filled fields). vLLM {vllm.__version__}.")

sample_labels = label_items(s300, "sample300")
save(sample_labels, "day4_ai_labels_OPEN_sample.csv", "label_id")
swap_labels = label_items(load_items("colab_input_swap.csv"), "swap")
save(swap_labels, "day4_ai_labels_OPEN_swap.csv", "label_id")

sample300: 0 cached, 300 to label
  finished 300 in 49 s
Saved day4_ai_labels_OPEN_sample.csv: 300 rows, 0 invalid (0.0%), 28 repaired (us_econ=0 with filled fields). vLLM 0.31.0.
swap: 0 cached, 100 to label
  finished 100 in 15 s
Saved day4_ai_labels_OPEN_swap.csv: 100 rows, 0 invalid (0.0%), 3 repaired (us_econ=0 with filled fields). vLLM 0.31.0.


## Step 2: the full corpus (about 20,000 items)
Run this only after section 4.9 passed. It is resumable: if Colab disconnects, reconnect, run the cells above (install, settings, prompt, server, labeler), then this one. Expect roughly 20 to 40 minutes on an A100 and a few hours on an L4.

In [9]:
corpus_labels = label_items(load_items("colab_input_corpus.csv"), "corpus")
save(corpus_labels, "day4_ai_labels_OPEN_corpus.csv", "item_id")
print(corpus_labels.ai_us_econ.value_counts().to_dict())

corpus: 0 cached, 20214 to label
  500/20214 done, 7.5 items/s
  1000/20214 done, 7.6 items/s
  1500/20214 done, 7.6 items/s
  2000/20214 done, 7.7 items/s
  2500/20214 done, 7.7 items/s
  3000/20214 done, 7.7 items/s
  3500/20214 done, 7.7 items/s
  4000/20214 done, 7.7 items/s
  4500/20214 done, 7.7 items/s
  5000/20214 done, 7.7 items/s
  5500/20214 done, 7.7 items/s
  6000/20214 done, 7.7 items/s
  6500/20214 done, 7.7 items/s
  7000/20214 done, 7.7 items/s
  7500/20214 done, 7.7 items/s
  8000/20214 done, 7.7 items/s
  8500/20214 done, 7.7 items/s
  9000/20214 done, 7.6 items/s
  9500/20214 done, 7.6 items/s
  10000/20214 done, 7.6 items/s
  10500/20214 done, 7.6 items/s
  11000/20214 done, 7.6 items/s
  11500/20214 done, 7.6 items/s
  12000/20214 done, 7.6 items/s
  12500/20214 done, 7.6 items/s
  13000/20214 done, 7.6 items/s
  13500/20214 done, 7.5 items/s
  14000/20214 done, 7.5 items/s
  14500/20214 done, 7.5 items/s
  15000/20214 done, 7.5 items/s
  15500/20214 done, 7.5 ite

In [ ]:
server.terminate()   # free the GPU when you are done